In [1]:
import torch 
import numpy as np 
import h5py
import os
from pathlib import Path
import importlib
import IPython.display as ipd
import torch.nn as nn
import src.spatial_attn_lightning as binaural_lightning 
import yaml
from pytorch_lightning import Trainer, seed_everything

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/scipy/__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


AttributeError: module 'torch' has no attribute 'set_float32_matmul_precision'

In [ ]:
### write method for taking average over last dimension of 3d tensor using torch einsum
# x is a 3d tensor
# weights is a 3d tensor of shape x.shape[0], x.shape[1], 1
# output is a 3d tensor of shape x.shape[0], x.shape[1], 1
# the einsum operation of "ijk,ijl->ijl" is equivalent to "torch.mean(x, dim=-1, keepdim=True)

print("Testing einsum method for taking average over last dimension of 4d tensor...")
# create a 3d tensor
x = torch.arange(24, dtype=torch.float).reshape(2,3,4).unsqueeze(0) # unsqueeze to add batch dim 
# create weights that will give average over last dimension of x
weights = torch.ones(2,3,4,1)
weights = weights / x.shape[-1]


# einsum method
ein_out = torch.einsum('bcft,cftl->bcfl', x, weights)
avg_method = torch.mean(x, dim=-1, keepdim=True)

torch.equal(ein_out, avg_method)

## Means learned way is to use torch parameter of shape (channels, frequency, 1) with einsum for learning weighted average for each channel and frequency bin 

Testing einsum method for taking average over last dimension of 4d tensor...


True

### Make sure this approach can actually learn the average operation 

In [3]:
#### Make sure this approach can learn an average 
# x = torch.arange(24, dtype=torch.float).reshape(2,3,4).unsqueeze(0) # unsqueeze to add batch dim 
x = torch.rand(1,2,3,4)
y =  torch.mean(x, dim=-1).squeeze()

# explicit weights 
weights = torch.ones(2,3,4,1)
weights = weights / x.shape[-1]

params = nn.Parameter(torch.ones(2,3,4,1))
# nn.init.constant_(params, x.shape[-1])
optimizer = torch.optim.Adam([params], lr=0.01)
loss = nn.MSELoss()

for i in range(100000):
    optimizer.zero_grad()
    out = torch.einsum('bcft,cftl->bcfl', x, params).squeeze()
    l = loss(out, y) #+ loss(params, weights)
    l.backward()
    optimizer.step()
    if i % 100 == 0:
        print(f"{l.item():.4f}")
    if torch.allclose(out, y):
        break


2.2496
0.0186


In [4]:
print(params.squeeze())
weights.squeeze()

tensor([[[0.2500, 0.2500, 0.2500, 0.2500],
         [0.2500, 0.2500, 0.2500, 0.2500],
         [0.2500, 0.2500, 0.2500, 0.2500]],

        [[0.2500, 0.2500, 0.2500, 0.2500],
         [0.2500, 0.2500, 0.2500, 0.2500],
         [0.2500, 0.2500, 0.2500, 0.2500]]], grad_fn=<SqueezeBackward0>)


tensor([[[0.2500, 0.2500, 0.2500, 0.2500],
         [0.2500, 0.2500, 0.2500, 0.2500],
         [0.2500, 0.2500, 0.2500, 0.2500]],

        [[0.2500, 0.2500, 0.2500, 0.2500],
         [0.2500, 0.2500, 0.2500, 0.2500],
         [0.2500, 0.2500, 0.2500, 0.2500]]])

In [5]:
explicit_out = torch.einsum('bcft,cftl->bcfl', x, weights).squeeze()
est_out = torch.einsum('bcft,cftl->bcfl', x, params).squeeze()

y, explicit_out, est_out

(tensor([[0.5741, 0.1857, 0.6440],
         [0.3775, 0.6343, 0.4195]]),
 tensor([[0.5741, 0.1857, 0.6440],
         [0.3775, 0.6343, 0.4195]]),
 tensor([[0.5741, 0.1857, 0.6440],
         [0.3775, 0.6343, 0.4194]], grad_fn=<SqueezeBackward0>))

In [9]:

config_path = "config/binaural_attn/word_task_half_co_loc_v08_gender_bal_4M_w_no_cue.yaml"
# config_path = "config/binaural_attn/word_task_half_co_loc_v06.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 0
config['hparas']['batch_size'] = 12
config['audio']['rep_kwargs']['rep_on_gpu'] = True
print(f"Default lr is {config['hparas']['lr']}")
config['hparas']['lr'] = 0.0001
print(f"Trying lr = {config['hparas']['lr']}")
config['model']['learned_gains'] = True 
config['model']['block_order'] = "LN -> Conv ->  ReLU" 
config['model']['v2_module'] = True

Default lr is 0.0001
Trying lr = 0.0001


In [10]:
# ## func to init layers with he initialization 
# def kaiming_init(model, mode='fan_in', init_type='normal'):
#     if init_type=='normal':
#         init_fn = torch.nn.init.kaiming_normal_
#     elif init_type=='uniform':
#         init_fn = torch.nn.init.kaiming_uniform_

#     for name, param in model.named_parameters():
#         if 'attn' in name:
#             continue
#         if name.endswith(".bias"):
#             # print(f"zero init {name}")
#             param.data.fill_(0)
#         elif any( part in name for part in ['conv', 'fullyconnected']):
#             # print(f"kaiming init {name}")
#             init_fn(param, mode=mode, nonlinearity='relu')



## Try simpler "global" time average

In [5]:
### write method for taking average over last dimension of 3d tensor using torch einsum

eg_time_dim = 4
out_dim = 1 

x = torch.arange(24, dtype=torch.float).reshape(2,3,4).unsqueeze(0) # unsqueeze to add batch dim 
weights = torch.ones((4,1))
weights = weights / x.shape[-1]
print(weights.shape)
# x is a 3d tensor

# einsum method
ein_out = torch.einsum('bcft,tl->bcfl', x, weights)
avg_method = torch.mean(x, dim=-1, keepdim=True)

torch.equal(ein_out, avg_method)

## Means learned way is to use torch parameter of shape (channels, frequency, 1) with einsum for learning weighted average for each channel and frequency bin 

torch.Size([4, 1])


True

In [8]:
### Test if we can optimize using this approach

#### Make sure this approach can learn an average 
# x = torch.arange(24, dtype=torch.float).reshape(2,3,4).unsqueeze(0) # unsqueeze to add batch dim 
x = torch.rand(1,2,3,4)
y =  torch.mean(x, dim=-1).squeeze()

# explicit weights 
weights = torch.ones(4,1)
weights = weights / x.shape[-1]

params = nn.Parameter(torch.ones(4,1))
# nn.init.constant_(params, x.shape[-1])
optimizer = torch.optim.Adam([params], lr=0.01)
loss = nn.MSELoss()

for i in range(100000):
    optimizer.zero_grad()
    out = torch.einsum('bcft,tl->bcfl', x, params).squeeze()
    l = loss(out, y) #+ loss(params, weights)
    l.backward()
    optimizer.step()
    if i % 1000 == 0:
        print(f"{l.item():.4f}")
    if torch.equal(out, y):
        break




2.5522
0.0000
0.0000
0.0000
0.0000
0.0000
0.0000


In [15]:
print(f"Hand constructed weights: {weights}")
print(f"Learned weights: {params.detach()}")

Hand constructed weights: tensor([[0.2500],
        [0.2500],
        [0.2500],
        [0.2500]])
Learned weights: tensor([[0.2500],
        [0.2500],
        [0.2500],
        [0.2500]])


In [9]:
explicit_out = torch.einsum('bcft,tl->bcfl', x, weights).squeeze()
est_out = torch.einsum('bcft,tl->bcfl', x, params).squeeze()

y, explicit_out, est_out

(tensor([[0.6045, 0.4469, 0.6910],
         [0.6426, 0.4075, 0.2826]]),
 tensor([[0.6045, 0.4469, 0.6910],
         [0.6426, 0.4075, 0.2826]]),
 tensor([[0.6045, 0.4469, 0.6910],
         [0.6426, 0.4075, 0.2826]], grad_fn=<SqueezeBackward0>))

## Test model implementation

In [11]:
seed_everything(0)
importlib.reload(binaural_lightning)
module = binaural_lightning.BinauralAttentionModule(config)

[rank: 0] Seed set to 0


Using explicit dim specification for demeaning in audio transforms
Using BinauralAuditoryAttentionCNNV2
v08 True
num_classes={'num_words': 800}
Model performing word task
Conv block order: LN -> Conv ->  ReLU
Using learned time averaged gains
coch_affine: True
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [12]:
# kaiming_init(module, mode='fan_in', init_type='uniform')

In [13]:
seed_everything(0)
trainer = Trainer(
    precision="32",
    limit_val_batches=0.0,
    num_nodes=1,
    # benchmark=True,
    devices=1, # was gpus=1,
    # detect_anomaly=True,
    # strategy="ddp_notebook",
    accelerator="gpu",
)
trainer.fit(module)

[rank: 0] Seed set to 0
/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /om2/user/imgriff/conda_envs/pytorch_2/lib/python3.1 ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                         | Params
------------------------------------------------------------------
0 | audio_transforms | AudioCompose                 | 0     
1 | model           

Using v06 dataset
/om/scratch/Sun/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v08
Using 0.1 cue free data
Using gender balanced training 4M set
cue type: mixed
mixture_percentages={'voice_only': 0.5, 'voice_and_location': 0.5}
913 files in train concat dataset


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=1` in the `DataLoader` to improve performance.


len training set = 304029
Epoch 0:   0%|          | 49/304029 [03:51<399:10:40,  0.21it/s, v_num=3.69e+7, train_loss=6.770, grad_norm=29.10] 

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/trainer/call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...


In [ ]:
!nvidia-smi

In [ ]:
## Check dataset 

dataset = module.dataset(**config['corpus'], mode='train')

In [ ]:
cue, target, distractor, labels = dataset[0]

In [ ]:
aud_transforms = module.audio_transforms

In [ ]:
ix = 1080

cue, target, distractor, labels = dataset[ix]

scene, _ = aud_transforms(target, distractor)

print("cue")
ipd.display(ipd.Audio(cue[0], rate=44100, normalize=False))
print("target")
ipd.display(ipd.Audio(target[0], rate=44100, normalize=False))
print("distractor")
ipd.display(ipd.Audio(distractor[0], rate=44100, normalize=False))
print("scene")
ipd.display(ipd.Audio(scene[0], rate=44100, normalize=False))